# Text Summarization using BART Transformer



In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
MODEL_SAVE_PATH = "/content/drive/MyDrive/auto_summarizer_model"
CHECKPOINT_PATH = "/content/drive/MyDrive/auto_summarizer_checkpoints"


In [3]:
!pip install datasets


In [4]:
from datasets import load_dataset

ds = load_dataset("knkarthick/dialogsum")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/11.3M [00:00<?, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [5]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [6]:
ds['train']

Dataset({
    features: ['id', 'dialogue', 'summary', 'topic'],
    num_rows: 12460
})

In [7]:
ds['train'][1]['dialogue']

"#Person1#: Hello Mrs. Parker, how have you been?\n#Person2#: Hello Dr. Peters. Just fine thank you. Ricky and I are here for his vaccines.\n#Person1#: Very well. Let's see, according to his vaccination record, Ricky has received his Polio, Tetanus and Hepatitis B shots. He is 14 months old, so he is due for Hepatitis A, Chickenpox and Measles shots.\n#Person2#: What about Rubella and Mumps?\n#Person1#: Well, I can only give him these for now, and after a couple of weeks I can administer the rest.\n#Person2#: OK, great. Doctor, I think I also may need a Tetanus booster. Last time I got it was maybe fifteen years ago!\n#Person1#: We will check our records and I'll have the nurse administer and the booster as well. Now, please hold Ricky's arm tight, this may sting a little."

In [8]:
#without fine-tuning
ds['train'][1]['summary']

'Mrs Parker takes Ricky for his vaccines. Dr. Peters checks the record and then gives Ricky a vaccine.'

In [9]:
!pip install transformers

In [10]:

from transformers import pipeline

pipe = pipeline("summarization", model="facebook/bart-large-cnn")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [12]:
article1=ds['train'][1]['dialogue']

In [13]:
pipe(article1,max_length=20,min_length=10,do_sample=False)

[{'summary_text': 'Ricky has received his Polio, Tetanus and Hepatitis B shots.'}]

In [14]:
#with fine-tuning model

In [15]:

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

In [16]:
#tokenization
def preprocess_func(batch):
  source=batch['dialogue']
  target=batch['summary']
  source_ids=tokenizer(source,truncation=True,padding='max_length',max_length=120)
  target_ids=tokenizer(target,truncation=True,padding='max_length',max_length=120)

  labels=target_ids['input_ids']
  labels=[[(label if label!= tokenizer.pad_token_id else -100)for label in labels_example]for labels_example in labels]

  return{
      "input_ids":source_ids["input_ids"],
      "attention_mask":source_ids["attention_mask"],
      "labels":labels
  }



In [17]:
df_source =ds.map(preprocess_func,batched=True)

Map:   0%|          | 0/12460 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [18]:
#training_argument
from transformers import TrainingArguments,Trainer

training_args=TrainingArguments(
    output_dir="/content",
    per_device_train_batch_size=8,
    report_to="none",
    num_train_epochs=2,
    logging_strategy="no",
    remove_unused_columns=True


)

In [19]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"

from accelerate.state import AcceleratorState
AcceleratorState._reset_state()


In [20]:


trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=df_source['train'],
    eval_dataset=df_source['test']


)



In [21]:
trainer.train()


Step,Training Loss


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Step,Training Loss


TrainOutput(global_step=3116, training_loss=1.2860520329799823, metrics={'train_runtime': 2775.4499, 'train_samples_per_second': 8.979, 'train_steps_per_second': 1.123, 'total_flos': 6328622658355200.0, 'train_loss': 1.2860520329799823, 'epoch': 2.0})

In [22]:
eval_results=trainer.evaluate()

In [23]:
eval_results

{'eval_loss': 1.6909739971160889,
 'eval_runtime': 46.006,
 'eval_samples_per_second': 32.604,
 'eval_steps_per_second': 4.086,
 'epoch': 2.0}

In [24]:
trainer.save_model(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)


('/content/drive/MyDrive/auto_summarizer_model/tokenizer_config.json',
 '/content/drive/MyDrive/auto_summarizer_model/special_tokens_map.json',
 '/content/drive/MyDrive/auto_summarizer_model/vocab.json',
 '/content/drive/MyDrive/auto_summarizer_model/merges.txt',
 '/content/drive/MyDrive/auto_summarizer_model/added_tokens.json',
 '/content/drive/MyDrive/auto_summarizer_model/tokenizer.json')

In [25]:
!ls /content/drive/MyDrive/auto_summarizer_model


config.json		model.safetensors	 tokenizer.json
generation_config.json	special_tokens_map.json  training_args.bin
merges.txt		tokenizer_config.json	 vocab.json


In [31]:
import torch

def summarize(text):

    device = "cuda" if torch.cuda.is_available() else "cpu"


    model.to(device)
    model.eval()


    inputs = tokenizer(
        text,
        truncation=True,
        max_length=1024,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}


    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            max_length=150,
            min_length=40,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True
        )


    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary


In [32]:
blog_post = """
As Yogi Berra famously said, it’s tough to make predictions, especially about the future. But had the baseball legend spent any time observing the UN climate negotiations, he could have safely predicted that climate finance will prove to be a key sticking point at COP29 in Baku at the end of this year.

‘Who will pay and how much?’ are perennial questions at the climate talks, but this year, the discussions about climate finance will be especially prominent. At COP29, Parties to the Paris Agreement must negotiate a new climate finance goal, to replace the existing commitment from 2009 for developed countries to provide US$100 billion climate finance annually from 2020 to 2025 - a commitment that only in 2022 was starting to be fulfilled, according to a recent OECD report.

It is vital that the forthcoming Bonn Climate Change Conference sends the right political signals, and lays the procedural and technical groundwork for an ambitious climate finance deal in Baku.

A pressing need

With global warming already destabilising the climate and devastating people’s lives and livelihoods, the need for finance to reduce greenhouse gas emissions and to adapt to a warming world has never been more pressing.

The sums involved are large. The Paris Agreement’s Global Stocktake process estimates that US$5.8-5.9 trillion is required to implement Nationally Determined Contributions (NDCs) in developing countries up to 2030. They will require US$215-387 billion annually over this period for adaptation. Investments of US$1.5 trillion in renewable energy are required worldwide every year up until 2030, according to IRENA.

But these sums are also affordable and beneficial for developed countries. They should be seen in the context of ongoing investments in energy and other infrastructure: around US$2.3 trillion was invested in energy infrastructure in 2023, of which US$1.74 trillion was in clean energy. These investments will generate strong returns for their investors and reduce the costs for energy consumers.

And, crucially, they should also be seen in the context of the alternative. The latest research estimates that the world economy is already set to face a 19% income reduction within the next 26 years based on the levels of warming we have already locked in. The more we delay and the more the planet heats, the greater the economic costs will be.

Laying the foundations for a new finance goal

While financial resources are beginning to flow, they are not flowing fast enough, and certainly not flowing to those developing countries where need is greatest and access to finance is most challenging.

The UN climate framework provides mechanisms that can enable those flows of climate finance. Back in 2015, parties at the climate talks agreed to establish a “new collective quantified goal” (NCQG) for climate finance. They agreed that the NCQG would be set prior to 2025.

The  ultimate size of the NCQG will be a product of the negotiations, but Parties have agreed it must be a significant increase from the floor of US$100 billion annually. For WWF, it must be needs-based and sufficiently ambitious to meet the scale of the challenge we face, and immediately accessible to help countries that are already facing the chaos of a destabilised climate system.

While developed countries are expected to provide financial and technical support, developing countries also have a role to play. Parties are due to submit revised NDCs in 2025, presenting how they plan to reduce emissions and adapt to climate change. Developing countries have the opportunity to use their NDCs to set out how international climate finance can support them and increase their ambition. To do this, they need to know the finance will be forthcoming.
"""

# Get the summary
summary = summarize(blog_post)
print("Summary:", summary)


Summary: #WWF thinks climate finance will be a key sticking point at COP29 in Baku, and it is vital that the forthcoming Bonn Climate Change Conference sends the right political signals and lays the groundwork for an ambitious climate finance deal. #WWF argues that developing countries need to use their NDCs to set out how international climate finance can support them and increase their ambition.
